# Моделювання газу твердих сфер

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from random import random

Константи

In [2]:
k = 1.38e-23

Вхідні дані

In [3]:
S = np.pi
H = 1
N = 2
R = 1e-6
M = 1e-6
T = 273
G = 9.8

Змінні та функції для зручності

In [4]:
# Діаметр основи циліднра
D = 2 * np.sqrt(S / np.pi)

# Випадковий одиничний вектор
def random_k():
    k = np.array([random() - 1, random() - 1, random() - 1])
    return k / np.linalg.norm(k)

In [5]:
# Кінетична енергія від швидкості
def K(v):
    return M*v**2/2

# Швидкість від кінетичної енергії
def v(K):
    return np.sqrt(2*K/M)

In [6]:
# Закон збереження імпульсу
def update_velocities(p1, p2, v1, v2):
    p = (p1 - p2) / np.linalg.norm(p1 - p2)
    d1 = v1 + p @ v1 * p
    d2 = v2 - p @ v2 * p
    v1new = d1 + (v1 + v2) @ p * p / 2 
    v2new = d2 - (v1 + v2) @ p * p / 2 
    return v1new, v2new

Генерування положення кульок (в межах посудини) і їхні швидкості випадковим чином, але так, 
щоб сума їхніх кінетичних енергії була рівна $3NkT/2$ (N--кількість частинок, T--температура, k--стала Больцмана)

In [7]:
max_enegry = 3 * N * k * T / 2
energies = np.array([random() * max_enegry for _ in range(N)])
energies /= sum(energies)

In [8]:
initial_positions = [np.array([(random() - 1) * D, (random() - 1) * D, (random() - 1) * H]) for _ in range(N)]
initial_velocities = [random_k() * v(K) for K in energies]

In [9]:
initial_positions = [np.array([-1, 0, 0]), np.array([1, 0, 0])]
initial_velocities = [np.array([1, 0, 0]), np.array([-1, 0, 0])]

Оновлення позицій 

In [10]:
def new_positions(positions, velocities, dt):
    return positions + velocities * dt

Перевірка на зіткнення

In [11]:
def detect_collisions(positions):
    collisions_matrix = np.zeros((N, N))
    for i, p1 in enumerate(positions[:-1]):
        for j, p2 in enumerate(positions[i+1:]):
            if np.linalg.norm(positions[i] - positions[i + j+1]) <= 2*R:
                collisions_matrix[i][i + j+1] = 1
                collisions_matrix[i + j+1][i] = 1
    return collisions_matrix

Оновлення швидкостей

In [12]:
from copy import deepcopy

def new_velocities(positions, collisions_matrix, velocities):
    new_velocities = deepcopy(velocities)
    for i in range(N):
        for j in range(N): 
            if collisions_matrix[i, j]:
                new = update_velocities(
                    positions[i],
                    positions[j],
                    velocities[i],
                    velocities[j])
                print(new)
                velocities[i][j], velocities[j][i] = new
    return new_velocities

### Cимуляція

Параметри симуляції

In [13]:
simulation_time = 10

Симуляція

In [14]:
t = 0
positions = {0: initial_positions}
velocities = {0: initial_velocities}

In [15]:
while t < simulation_time:
   current_positions = list(positions.values())[-1]
   current_velocities = list(velocities.values())[-1]
   dt = R / 4 / max(current_velocities, key=np.linalg.norm)
   updated_positions = new_positions(current_positions, current_velocities, dt)
   collisions_matrix = detect_collisions(updated_positions)
   updated_velocities = new_velocities(updated_positions, collisions_matrix, current_velocities)
   t += dt
   positions[t] = updated_positions 
   velocities[t] = updated_velocities

/tmp/ipykernel_34130/2152821456.py:4: RuntimeWarning: divide by zero encountered in divide
  dt = R / 4 / max(current_velocities, key=np.linalg.norm)
/tmp/ipykernel_34130/44288031.py:2: RuntimeWarning: invalid value encountered in multiply
  return positions + velocities * dt


TypeError: unhashable type: 'numpy.ndarray'

### а) графік розподілу частинок по швидкостях (розподіл Максвела)

In [ ]:
list(positions.values())[-1]

[array([-0.73350309, -0.41719272, -0.14376806]),
 array([-1.5931845 , -1.76216225, -0.35551724]),
 array([-0.88308764, -0.77321655, -0.99330999]),
 array([-0.58367101, -0.03961708, -0.16966573]),
 array([-1.64158018, -1.24024957, -0.24681324]),
 array([-0.66819634, -1.34014929, -0.20045128]),
 array([-0.38406837, -1.20601131, -0.51285728]),
 array([-1.61414208, -0.20807229, -0.37766297]),
 array([-0.06749053, -1.26835781, -0.40145116]),
 array([-1.36481298, -1.36706279, -0.9511098 ]),
 array([-1.96396958, -1.23892512, -0.2946193 ]),
 array([-1.48503275, -1.05024435, -0.90620818]),
 array([-0.56383296, -0.85375774, -0.7525335 ]),
 array([-0.91123535, -1.53380225, -0.77223173]),
 array([-1.7236036 , -1.08094274, -0.91396456]),
 array([-0.4874983 , -1.29896273, -0.46495319]),
 array([-0.44765784, -1.77136124, -0.91098251]),
 array([-0.7612387 , -0.26046736, -0.01845045]),
 array([-0.44767264, -1.25873807, -0.76347034]),
 array([-1.11544632, -1.91098925, -0.60751238]),
 array([-0.02604599,

In [ ]:
positions = [1,2,3,4,5,6,7,8,9,10]
for i, p1 in enumerate(positions[:-1]):
    for j, p2 in enumerate(positions[i+1:]):
        print(i+j+ 1)

1
2
3
4
5
6
7
8
9
2
3
4
5
6
7
8
9
3
4
5
6
7
8
9
4
5
6
7
8
9
5
6
7
8
9
6
7
8
9
7
8
9
8
9
9


In [ ]:
positions[1:]

[2, 3, 4, 5, 6, 7, 8, 9, 10]